In [0]:
import pyspark.sql.functions as F

# ── Read bronze ────────────────────────────────────────────────────────────
df_silver_raw = spark.read.table("bronze.sales_raw")

# ── Deduplicate before explode ─────────────────────────────────────────────
df_silver_exploded = (
    df_silver_raw
    .dropDuplicates(["order_number"])
    .withColumn("product", F.explode("ordered_products"))
)

# ── Extract all fields ─────────────────────────────────────────────────────
df_silver = (
    df_silver_exploded
    # top level order fields
    .withColumn("order_number",   F.col("order_number"))
    .withColumn("customer_id",    F.col("customer_id"))
    .withColumn("customer_name",  F.col("customer_name"))   
    .withColumn("order_datetime", F.when(F.col("order_datetime") != "", F.to_timestamp(F.from_unixtime(F.col("order_datetime").cast("bigint")))).otherwise(None))
    # product struct fields
    .withColumn("product_id",     F.col("product.id"))
    .withColumn("product_name",   F.col("product.name"))
    .withColumn("price",          F.col("product.price").cast("double"))
    .withColumn("qty",            F.col("product.qty").cast("int"))
    .withColumn("currency",       F.col("product.curr"))
    .withColumn("unit",           F.col("product.unit"))
    # promotion fields
    .withColumn("promo_id",       F.col("product.promotion_info.promo_id"))
    .withColumn("promo_disc",     F.col("product.promotion_info.promo_disc"))
    .drop("ordered_products", "product", "promo_info", "clicked_items")
)

# ── Write to silver ────────────────────────────────────────────────────────
(df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.sales_clean"))

# ── Verify ─────────────────────────────────────────────────────────────────
df_check = spark.read.table("silver.sales_clean")
df_check.printSchema()
print("bronze rows :", df_silver_raw.count())
print("silver rows :", df_check.count())
df_check.show(5, truncate=False)




spark.read.table("silver.sales_clean").show(5, truncate=False)

In [0]:

%sql

SELECT * FROM workspace.silver.sales_clean LIMIT 5